# Embeddings with `affon:embed`

This notebook shows both local embedding providers currently exposed by `affon:embed`:
- `model2vec` for lightweight pooled text embeddings
- `onnx` for transformer-backed text embeddings and contextual token embeddings

Notes:
- Models download and cache locally on first use.
- `embedText()` returns one pooled vector per input text.
- `embedTokens()` is currently available only for `provider: 'onnx'`.

## Setup

In [1]:
import { createEmbedder } from 'affon:embed'



function cosine(a: number[], b: number[]): number {
  let dot = 0
  let normA = 0
  let normB = 0

  for (let i = 0; i < a.length; i++) {
    dot += a[i] * b[i]
    normA += a[i] * a[i]
    normB += b[i] * b[i]
  }

  return dot / (Math.sqrt(normA) * Math.sqrt(normB))
}

## Model2Vec Provider

Use this when you want fast local text embeddings with the current Model2Vec-backed runtime.

In [2]:
const model2vec = createEmbedder({
  provider: 'model2vec',
  model_name: 'minishlab/potion-base-8M',
})

console.log('model2vec dim:', model2vec.dim)

model2vec dim: 256


In [3]:
const model2vecBatch = model2vec.embedText([
  'The cat is sleeping on the sofa.',
  'A kitten is resting on the couch.',
  'The stock market closed higher today.',
])

const model2vecRows = model2vecBatch.to_array() as number[][]
console.log('model2vec shape:', model2vecBatch.shape)
console.log('cat vs kitten:', cosine(model2vecRows[0], model2vecRows[1]).toFixed(4))
console.log('cat vs market:', cosine(model2vecRows[0], model2vecRows[2]).toFixed(4))

model2vec shape: [ 3, 256 ]
cat vs kitten: 0.7143
cat vs market: -0.0290


## ONNX Provider

Use this when you want transformer-backed embeddings. This provider supports both pooled text embeddings and contextual token embeddings.

In [4]:
const onnx = createEmbedder({
  provider: 'onnx',
  model_name: 'AllMiniLML6V2',
})

console.log('onnx dim:', onnx.dim)

onnx dim: 384


In [5]:
const onnxBatch = onnx.embedText([
  'query: The cat is sleeping on the sofa.',
  'passage: A kitten is resting on the couch.',
  'passage: The stock market closed higher today.',
])

const onnxRows = onnxBatch.to_array() as number[][]
console.log('onnx shape:', onnxBatch.shape)
console.log('query vs couch:', cosine(onnxRows[0], onnxRows[1]).toFixed(4))
console.log('query vs market:', cosine(onnxRows[0], onnxRows[2]).toFixed(4))

onnx shape: [ 3, 384 ]
query vs couch: 0.7072
query vs market: 0.0553


In [6]:
const tokenVectors = onnx.embedTokens('query: hello world')
console.log('token shape:', tokenVectors.shape)
console.log(tokenVectors)

token shape: [ 6, 384 ]
[6×384 Tensor, dtype=f32]
             0         1          2          3          4          5    ⋯         378        379        380        381        382       383
  ┌                                                                                                                                          ┐
0 │   0.210083   0.20443  -0.055242   0.385848  -0.349067  -0.424624    ⋯    0.131524   0.188797   0.026033  -0.217474   -0.05936  0.064731  │
1 │   0.818167  0.469569   0.223819   0.628091  -0.971932   0.343026    ⋯   -0.208793  -0.289455   0.520529  -0.711829   -0.89135  1.113875  │
2 │   0.145079  0.568501  -0.403878    0.57657  -0.740321  -0.539219    ⋯    0.134229  -0.101673   0.657219  -0.128604   0.096282  0.904602  │
3 │  -0.221019  0.396734   0.488511    0.21047  -0.727901  -0.750649    ⋯    -0.34729   0.150703   0.443838   0.639044   0.166184  0.533796  │
4 │   0.291722  0.388473  -0.422048  -0.198195   0.291728  -1.622971    ⋯    0.219353   0.55694

## Summary

- `model2vec.embedText(...)` gives fast pooled text embeddings
- `onnx.embedText(...)` gives pooled transformer embeddings
- `onnx.embedTokens(...)` gives contextual token embeddings with shape `[seq_len, dim]`